In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
sample_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/sample_submission.csv")

In [ ]:
train_df

In [ ]:
train_df.info()

In [ ]:
sample_df

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
#　SalePriceを対数変換
X = train_df
y = np.log1p(train_df["SalePrice"])
X_test = test_df

In [ ]:
#SalePriceとの相関を調べる
#＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝
#↓計算させるコード分かんないのでAIにおまかせ↓

#相関係数の計算
# 1. 数値特徴量のみを抽出(元文字列データは01に変換されてるだけなので相関はあんまり関係なさそう？)
# pandasのselect_dtypesで数値型（int64とfloat64）の列を抽出
numerical_features = train_df.select_dtypes(include=np.number)

# 2. SalePriceとの相関係数を計算
# 相関行列を計算し、'SalePrice'列を取り出す
correlations = numerical_features.corr()['SalePrice']

# 3. 'SalePrice'自身との相関（常に1）を除外
correlations = correlations.drop('SalePrice')

# 4. 相関の絶対値が高い順にソートして表示
# abs()で絶対値を取り、sort_values(ascending=False)で降順にソート
sorted_correlations = correlations.abs().sort_values(ascending=False)

print("--- SalePriceとの相関（絶対値）が高い数値特徴量ランキング ---")
print(sorted_correlations.head(30)) # 上位30個を表示

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#一番相関大きい”OverallQual”をヒストグラム化
plt.hist(X["OverallQual"])

**OverallQual（総合評価）：1~10点評価なので外れ値処理はしないでよさそう**

In [ ]:
#二番目に相関大きい”GrLivArea”の散布図作成
plt.figure(figsize=(10, 6))
plt.scatter(x=X['GrLivArea'], y=y)
plt.xlabel('GrLivArea')
plt.ylabel('Log(SalePrice)')
plt.title('Log(SalePrice) vs GrLivArea Scatter Plot')
plt.show()

**GrLivArea (居住面積)　： 面積4000以上、かつ、対数変換後のSalePriceが12.5以下のデータが外れ値
→除去**

In [ ]:
# 外れ値の除去 (GrLivArea > 4000 かつ ｙ(log(SalePrice)) < 12.5程度の物件2件)
outliers_to_remove = X[(X['GrLivArea'] > 4000) & (y < 12.5)].index

# 訓練データと目的変数から該当する行を削除
X = X.drop(outliers_to_remove)
y = y.drop(outliers_to_remove)


In [ ]:
# One-Hot Encodingするために一旦学習データとテストデータを結合
#↑学習データとテストデータで分けてやったら、最後にエラーでちゃったから
X_combined = pd.concat([X, X_test], ignore_index=True)

In [ ]:
#欠損値の処理

# 欠損値の合計数と割合を計算し、割合の大きい順に並べ替える
total_missing = train_df.isnull().sum().sort_values(ascending=False)
percent_missing = (train_df.isnull().sum() / train_df.isnull().count()).sort_values(ascending=False)

# 欠損値が1つでもある特徴量だけを抽出
missing_data = pd.concat([total_missing, percent_missing], axis=1, keys=['欠損数', '欠損割合'])
missing_data_head = missing_data[missing_data['欠損数'] > 0]

print("--- 欠損値の多い特徴量---")
print(missing_data_head)

In [ ]:
#欠損値が「設備がない」を意味するものをNoneに置き替える

X_combined[['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 
    'MasVnrType']]= X_combined[['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 
    'MasVnrType']].fillna("None")

In [ ]:
#　置き換わったか確認
X_combined["Alley"]

In [ ]:
#数値変数の欠損を0に置き換える

X_combined[['GarageYrBlt', 'MasVnrArea', 
    ]]= X_combined[['GarageYrBlt', 'MasVnrArea', 
    ]].fillna(0)

In [ ]:
#　その他欠損値の処理
#　外れ値の影響を受けないように平均値を入れる
X_combined["LotFrontage"] = X_combined["LotFrontage"].fillna(X_combined["LotFrontage"].mean())

In [ ]:
print("X_combined の欠損数:\n", X_combined.isnull().sum()[X_combined.isnull().sum() > 0])

In [ ]:
#新しい変数の作成

# 1. 総面積 (TotalSF) の作成
X_combined['TotalSF'] = (
    X_combined['TotalBsmtSF'] +  # 地下室の総面積
    X_combined['1stFlrSF'] +     # 1階の面積
    X_combined['2ndFlrSF']       # 2階の面積
)


# 2. 築年数 (Age) の作成
# YrSold (販売年) - YearBuilt (建設年)
X_combined['Age'] = X_combined['YrSold'] - X_combined['YearBuilt']
 #データエラー（築年数が負の値）があれば0に修正
X_combined['Age'] = X_combined['Age'].apply(lambda x: x if x > 0 else 0)


print("--- 新しい変数の確認（最初の5行） ---")
print(X_combined[['TotalSF', 'Age']].head())

print(f"\n✅ 2つの新しい特徴量を作成し、特徴量数は {X_combined.shape[1]} 個になりました。")

In [ ]:
#地域情報の変数を作成

# 1. 訓練データ（外れ値除去後）を抽出
train_size = len(y)
X_train_clean = X_combined.iloc[:train_size, :].copy()

# 2. 地域ごとの平均価格を計算 (訓練データ全体を使用)
# これは平均値エンコーディングの基本形です。
neighborhood_mean_prices = y.groupby(X_train_clean['Neighborhood']).mean()

# 3. 新しい特徴量 'Neighborhood_Price' を作成
X_combined['Neighborhood_Price'] = X_combined['Neighborhood'].map(neighborhood_mean_prices)

# 4. テストデータで欠損が出た場合の処理 (訓練データに存在しない地域の場合)
# 欠損値は全体の平均価格で埋めることで、過学習を防ぐ
global_mean_price = y.mean()
X_combined['Neighborhood_Price'] = X_combined['Neighborhood_Price'].fillna(global_mean_price)

print("✅ 地域別平均価格の特徴量 'Neighborhood_Price' が安全に追加されました。")
print(X_combined[['Neighborhood', 'Neighborhood_Price']].head())

In [ ]:
# --- ターゲットエンコーディングの実行 ---　失敗、、、

#from sklearn.model_selection import KFold

# 1. 訓練データと全データを再び分割し直す
# 以前の処理でX_combinedは全データ、yは対数SalePrice（外れ値除去済み）です。
#train_size = len(y)
#X_train_encoded = X_combined.iloc[:train_size, :].copy()
#X_test_encoded = X_combined.iloc[train_size:, :].copy()
#y_train_log = y.copy()

# 2. ターゲットエンコーディングのためのKFold分割を設定
#kf = KFold(n_splits=5, shuffle=True, random_state=42)
#X_train_encoded['Neighborhood_Target'] = 0 # 新しい列を初期化

# 3. KFold交差検証を使って地域別の平均価格を計算 (過学習対策)
#for train_index, valid_index in kf.split(X_train_encoded, y_train_log):
#    # 訓練用Foldのデータを取得
#    X_fold_train = X_train_encoded.iloc[train_index]
#    y_fold_train = y_train_log.iloc[train_index]
    
    # 検証用Foldのデータを取得
#    X_fold_valid = X_train_encoded.iloc[valid_index]

    # 訓練用Foldの地域別の平均価格 (対数SalePrice) を計算
#    target_mean = y_fold_train.groupby(X_fold_train['Neighborhood']).mean()
    
    # 検証用Foldの対応する地域に平均値をマッピング
#    X_train_encoded.loc[valid_index, 'Neighborhood_Target'] = X_fold_valid['Neighborhood'].map(target_mean)

# 4. テストデータへの適用
# テストデータには、訓練データ全体の地域別平均値を適用する
#global_mean = y_train_log.groupby(X_train_encoded['Neighborhood']).mean()
#X_test_encoded['Neighborhood_Target'] = X_test_encoded['Neighborhood'].map(global_mean)

In [ ]:
# --- 歪度の高い特徴量の特定と対数変換 ---　　失敗、、、

# 数値特徴量を抽出（IDとSalePriceを除く）
#numerical_features = X_combined.select_dtypes(include=['int64', 'float64']).columns.tolist()
# 既に削除されている可能性のある列を除外
#if 'Id' in numerical_features:
#    numerical_features.remove('Id')
#if 'SalePrice' in numerical_features:
#    numerical_features.remove('SalePrice')

# 歪度を計算し、0.5より大きい特徴量に絞る
#from scipy.stats import skew
#skewness = X_combined[numerical_features].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
#skewed_features = skewness[skewness > 0.5].index

# 歪度の高い特徴量に対数変換を適用 (np.log1p)
# これが線形モデルの性能向上に非常に重要です
#for feat in skewed_features:
#    X_combined[feat] = np.log1p(X_combined[feat])

#print(f"\n✅ {len(skewed_features)}個の数値特徴量に歪度補正（対数変換）を適用しました。")

In [ ]:
# --- 順序カテゴリ変数の数値化 ---

# マッピング辞書の定義 (data_description.txtを参照)
qual_mapping = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0, 'NA': 0}
exposure_mapping = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0, 'NA': 0}

# 順序エンコーディングを行う列
qual_cols_to_encode = [
    'ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual', 
    'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC',
    'BsmtQual', 'BsmtCond'
]

for col in qual_cols_to_encode:
    # 新しい列（例: ExterQual_Num）を作成し、マッピングを適用
    X_combined[col + '_Num'] = X_combined[col].map(qual_mapping).fillna(0)
    # 元の文字列列はOne-Hot Encodingで処理されないよう、ここではそのまま残します (後のget_dummiesが対応)

# BsmtExposure の特別処理
X_combined['BsmtExposure_Num'] = X_combined['BsmtExposure'].map(exposure_mapping).fillna(0)
print(f"✅ 順序カテゴリ変数に数値エンコーディングを適用しました。")

In [ ]:
# train_df に基づいて欠損値の割合が高い特徴量を特定
# 欠損値の割合が 60% を超える特徴量を削除対象とします
percent_missing = (train_df.isnull().sum() / len(train_df))
high_missing_features = percent_missing[percent_missing > 0.60].index.tolist()

# 削除対象のリストに追加（今回はそのまま削除対象とする）
features_to_drop = high_missing_features

In [ ]:
#追加した変数を踏まえてもう1度相関係数を計算
# 1. 数値特徴量のみを抽出(元文字列データは01に変換されてるだけなので相関はあんまり関係なさそう？)
# pandasのselect_dtypesで数値型（int64とfloat64）の列を抽出
numerical_features = X_combined.select_dtypes(include=np.number)

# 2. SalePriceとの相関係数を計算
# 相関行列を計算し、'SalePrice'列を取り出す
correlations = numerical_features.corr()['SalePrice']

# 3. 'SalePrice'自身との相関（常に1）を除外
correlations = correlations.drop('SalePrice')

# 4. 相関の絶対値が高い順にソートして表示
# abs()で絶対値を取り、sort_values(ascending=False)で降順にソート
sorted_correlations = correlations.abs().sort_values(ascending=False)

print("--- SalePriceとの相関（絶対値）が高い数値特徴量ランキング ---")
print(sorted_correlations.head(30)) # 上位30個を表示

In [ ]:
# 相関の絶対値が 0.30 未満の特徴量を削除対象とする
#numerical_features は train_dfの数値列
correlations = numerical_features.corr()['SalePrice'].abs().drop('SalePrice')
low_corr_features = correlations[correlations < 0.30].index.tolist()

# 削除対象のリストに追加
features_to_drop.extend(low_corr_features)

# 重複を排除し、最終的な削除リストを確定
features_to_drop = list(set(features_to_drop))

In [ ]:
# --- 上記で削除対象としたものを実際に削除 ---

# X_combined = pd.concat([X, X_test], ignore_index=True) を実行した後:
# SalePriceも削除（errors='ignore'で、存在しない場合は無視）
X_combined = X_combined.drop(['SalePrice'] + features_to_drop, axis=1, errors='ignore')

#残った特徴量を表示
print(X_combined)

In [ ]:
# One-Hot Encoding（文字列データ→数値データ）
X_combined_encoded = pd.get_dummies(X_combined)

In [ ]:
# X の元の行数
train_size = len(X) 

# エンコード後のデータセットを分割
X_encoded = X_combined_encoded[:train_size]      # 学習データ
X_test_encoded = X_combined_encoded[train_size:] # テストデータ

In [ ]:
# 🌟 もう1回欠損値処理: すべての NaN を 0 で埋める 🌟
# これにより、未処理の数値欠損値や、One-Hot Encodingで発生したNaNも処理される
X_encoded = X_encoded.fillna(0)
X_test_encoded = X_test_encoded.fillna(0)

In [ ]:
print(f"\nX_encoded: {X_encoded.shape}")      # (1460, 約300)
print(f"X_test_encoded: {X_test_encoded.shape}") # (1459, 約300)

In [ ]:
# 5. One-Hot Encodingの実行　（KFoldで地域情報作成した場合のやり方）
# 新しいデータフレームX_combined_encodedを作成し直す

# Neighborhood列はそのまま残し、get_dummiesで処理させます
#X_train_encoded.drop(['Neighborhood'], axis=1, errors='ignore', inplace=True)
#X_test_encoded.drop(['Neighborhood'], axis=1, errors='ignore', inplace=True)

# 訓練データとテストデータを再結合し、One-Hot Encoding
#X_combined_final = pd.concat([X_train_encoded, X_test_encoded], ignore_index=True)
#X_combined_encoded = pd.get_dummies(X_combined_final)

# データを再分割
#train_size = len(y_train_log)
#X_encoded = X_combined_encoded.iloc[:train_size, :].copy()
#X_test_encoded = X_combined_encoded.iloc[train_size:, :].copy()

# 最終的な欠損値処理
#X_encoded = X_encoded.fillna(0)
#X_test_encoded = X_test_encoded.fillna(0)

#print("✅ 地域別平均価格エンコーディングが完了しました。")
#print(f"新しい特徴量 'Neighborhood_Target' の最初の5行:\n{X_encoded['Neighborhood_Target'].head()}")

### モデル構築


* ★　Lasso 回帰 (L1 正則化)
* Lasso 回帰は、係数の絶対値の和を罰則項として使用します。
* 罰則項: 係数の絶対値の和効果: 重要度の低い特徴量の係数を完全にゼロにすることです。
* 特徴: 特徴量選択の役割を果たします。自動的に不要な特徴量をモデルから除外するため、モデルがより解釈しやすくなります。
* 　　　特徴量が非常に多い（例：One-Hot Encoding後）場合にオススメ！モデルがシンプルになり、解釈性が向上！


In [ ]:
# ----------------------------------------------------------------------------------
# 【追加1】StandardScalerとPipelineのインポート (コードの冒頭部分に追加)
# ----------------------------------------------------------------------------------
#from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Lasso
#from sklearn.preprocessing import StandardScaler
#from sklearn.pipeline import make_pipeline

In [ ]:
#from sklearn.linear_model import LinearRegression

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X_encoded, y, random_state=0)

In [ ]:
model = Lasso(alpha=0.0007)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
#Lasso回帰モデルでどの特徴量を削除したのか、最終的にどの特徴量が残っているのかを確認！
#＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝＝
#↓確認方法が分かんないのでAIにおまかせ↓


# 1. 訓練後の係数（重み）をデータフレームとして取得
#    X_train.columnsは、モデルが訓練に使用した特徴量の名前のリストです。
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient':model.coef_
})

# 2. 係数の絶対値に基づいて降順にソート (重要度の高い順)
coef_df['Abs_Coefficient'] = np.abs(coef_df['Coefficient'])
coef_df = coef_df.sort_values(by='Abs_Coefficient', ascending=False)

# 3. 係数が完全にゼロ、または非常にゼロに近い特徴量を分離する
#    Lassoは厳密にゼロにしますが、浮動小数点誤差を考慮して非常に小さな閾値を使うこともあります。
#    ここでは簡単のために、係数がゼロである特徴量に絞ります。

zero_threshold = 1e-10 # 非常に小さい数（ほぼゼロと見なす閾値）

# --- 削除された特徴量（Lassoが自動的に係数をゼロにしたもの） ---
features_dropped = coef_df[coef_df['Abs_Coefficient'] < zero_threshold]
print("--- 📉 Lassoによって係数がゼロになった特徴量 (自動削除) ---")
print(f"合計 {len(features_dropped)} 個が削除されました。")
print(features_dropped[['Feature', 'Coefficient']].head(10)) 

# --- 残った特徴量（モデルが使用したもの） ---
features_retained = coef_df[coef_df['Abs_Coefficient'] >= zero_threshold]
print("\n--- 📈 モデルに残った特徴量 (重要度の高い順) ---")
print(f"合計 {len(features_retained)} 個が使用されました。")
print(features_retained[['Feature', 'Coefficient']].head(10)) 

### モデル適用と評価

In [ ]:
pred_train = model.predict(X_train)
pred_valid = model.predict(X_valid)
pred_test = model.predict(X_test_encoded)

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
print(mean_squared_error(y_train, pred_train), mean_squared_error(y_valid, pred_valid))

### Submission

In [ ]:
sub_df = pd.DataFrame({
    "Id": sample_df["Id"], 
    "SalePrice": np.expm1(pred_test)  # logを元に戻す
})

In [ ]:
sub_df.to_csv("simple.csv", index=False, float_format='%.8f')

In [ ]:
sub_df